<a href="https://colab.research.google.com/github/arzmtva/my-data-analysis/blob/main/Practical_6_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from google.colab import files

uploaded = files.upload()


Saving archive (2).zip to archive (2).zip


In [2]:

import pandas as pd
import os

filename = next(iter(uploaded.keys()))

if filename.endswith(".csv"):
    df = pd.read_csv(filename)
elif filename.endswith((".xlsx", ".xls")):
    df = pd.read_excel(filename)
elif filename.endswith(".txt"):
    df = pd.read_csv(filename, sep="\t")
else:
    raise ValueError("Загрузите файл CSV, XLSX, XLS или TXT")

print("Размер датасета:", df.shape)
print("\nНазвания столбцов:")
print(df.columns.tolist())
print("\nПервые 5 строк:")
display(df.head())



ValueError: Загрузите файл CSV, XLSX, XLS или TXT

In [3]:

print(uploaded.keys())


dict_keys(['archive (2).zip'])


In [4]:

import pandas as pd
import os
import zipfile
import json

filename = next(iter(uploaded.keys()))

# Если файл — ZIP, распаковываем его
if zipfile.is_zipfile(filename):
    with zipfile.ZipFile(filename, "r") as archive:
        candidates = [
            f for f in archive.namelist()
            if not f.startswith("__MACOSX/")
            and not os.path.basename(f).startswith(".")
            and f.lower().endswith(
                (".csv", ".xlsx", ".xls", ".txt", ".json")
            )
        ]

        if not candidates:
            raise ValueError(
                "В архиве не найден CSV, Excel, TXT или JSON файл. "
                "Проверь содержимое архива."
            )

        print("Файлы в архиве:", candidates)
        selected = candidates[0]
        print("Загружаем:", selected)

        with archive.open(selected) as file:
            if selected.lower().endswith(".csv"):
                df = pd.read_csv(file)
            elif selected.lower().endswith((".xlsx", ".xls")):
                df = pd.read_excel(file)
            elif selected.lower().endswith(".txt"):
                df = pd.read_csv(file, sep=None, engine="python")
            else:
                df = pd.read_json(file)

elif filename.lower().endswith(".csv"):
    df = pd.read_csv(filename)
elif filename.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(filename)
elif filename.lower().endswith(".txt"):
    df = pd.read_csv(filename, sep=None, engine="python")
elif filename.lower().endswith(".json"):
    df = pd.read_json(filename)
else:
    raise ValueError(
        f"Неподдерживаемый файл: {filename}. "
        "Посмотри его название и расширение."
    )

print("\nРазмер датасета:", df.shape)
print("\nНазвания столбцов:")
print(df.columns.tolist())
print("\nПервые 5 строк:")
display(df.head())


Файлы в архиве: ['govt_fund_utilization.csv']
Загружаем: govt_fund_utilization.csv

Размер датасета: (1000, 21)

Названия столбцов:
['Record_ID', 'Financial_Year', 'Date_of_Fund_Release', 'State', 'District', 'Department', 'Scheme_Name', 'Scheme_Code', 'Fund_Source', 'Implementing_Agency', 'Sanctioned_Amount_Lakhs', 'Fund_Released_Lakhs', 'Fund_Utilized_Lakhs', 'Utilization_Percent', 'Physical_Target_Units', 'Physical_Achievement_Units', 'Beneficiaries_Target', 'Beneficiaries_Covered', 'Status', 'Audit_Remarks', 'Nodal_Officer_Contact']

Первые 5 строк:


,Record_ID,Financial_Year,Date_of_Fund_Release,State,District,Department,Scheme_Name,Scheme_Code,Fund_Source,Implementing_Agency,...,Fund_Released_Lakhs,Fund_Utilized_Lakhs,Utilization_Percent,Physical_Target_Units,Physical_Achievement_Units,Beneficiaries_Target,Beneficiaries_Covered,Status,Audit_Remarks,Nodal_Officer_Contact
0,GFU00643,2022-23,2021-08-25,West Bengal,Siliguri,Ministry of Agriculture & Farmers Welfare,PM Kisan Samman Nidhi,PMKISAN,Central Sector,Gram Panchayat,...,517.65,212.25,41.00,427,38.0,4138,344.0,Delayed,Awaiting third-party audit clearance,Sunita Yadav - District Collector Office - sun...
1,GFU00763,2021-22,2023-04-16,Tamil Nadu,Salem,Ministry of Agriculture & Farmers Welfare,Pradhan Mantri Fasal Bima Yojana,PMFBY,Central Sector,State Nodal Agency,...,44.61,27.25,61.08,367,104.0,4988,1373.0,Delayed,Material procurement in progress,Rakesh Yadav - Block Development Officer - rak...
2,GFU00910,2023-24,2023-01-31,Kerala,Thiruvananthapuram,Ministry of Health & Family Welfare,National Health Mission,NHM,Centrally Sponsored (60:40),District Panchayat / Zilla Parishad,...,634.28,NaN,NaN,6,5.0,2151,1621.0,Delayed,Utilization certificate submitted and verified,Rakesh Reddy - District Collector Office - rak...
3,GFU00200,2023-24,2024-08-29,West Bengal,Asansol,Ministry of Education,PM Poshan (Mid-Day Meal Scheme),PMPOSHAN,Centrally Sponsored (60:40),NGO / Partner Organisation,...,247.44,147.35,59.55,55,3.0,4122,213.0,Ongoing,Fund utilization certificate pending,Anjali Sharma - Assistant Engineer - anjali.sh...
4,GFU00587,2021-22,2021-06-21,Karnataka,Belagavi,Ministry of Rural Development,Pradhan Mantri Awas Yojana - Gramin,PMAY-G,Centrally Sponsored (60:40),District Panchayat / Zilla Parishad,...,593.82,451.45,76.02,406,337.0,2984,2565.0,Delayed,Fund utilization certificate pending,Kavita Das - Scheme Nodal Officer - kavita.das...


In [5]:

# Оставляем текст замечаний и категорию статуса
text_column = "Audit_Remarks"
label_column = "Status"

df = df.dropna(subset=[text_column, label_column]).copy()

df[text_column] = df[text_column].astype(str)
df[label_column] = df[label_column].astype(str)

print("Количество записей:", len(df))
print("\nКатегории для классификации:")
print(df[label_column].value_counts())

print("\nПримеры текстов:")
display(df[[text_column, label_column]].head())


Количество записей: 930

Категории для классификации:
Status
Completed      329
Ongoing        304
Delayed        197
Not Started    100
Name: count, dtype: int64

Примеры текстов:


,Audit_Remarks,Status
0,Awaiting third-party audit clearance,Delayed
1,Material procurement in progress,Delayed
2,Utilization certificate submitted and verified,Delayed
3,Fund utilization certificate pending,Ongoing
4,Fund utilization certificate pending,Delayed


In [6]:

import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"_+", " ", text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

df["clean_text"] = df[text_column].apply(clean_text)

display(df[[text_column, "clean_text"]].head())


,Audit_Remarks,clean_text
0,Awaiting third-party audit clearance,awaiting third party audit clearance
1,Material procurement in progress,material procurement progress
2,Utilization certificate submitted and verified,utilization certificate submitted verified
3,Fund utilization certificate pending,fund utilization certificate pending
4,Fund utilization certificate pending,fund utilization certificate pending


In [8]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["clean_text"]
y = df[label_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 1)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Обучающих текстов:", len(X_train))
print("Тестовых текстов:", len(X_test))
print("Размер TF-IDF матрицы:", X_train_tfidf.shape)
print("Размер словаря:", len(vectorizer.vocabulary_))


Обучающих текстов: 744
Тестовых текстов: 186
Размер TF-IDF матрицы: (744, 34)
Размер словаря: 34


In [9]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


Accuracy: 0.4032

Classification report:
              precision    recall  f1-score   support

   Completed       0.39      0.62      0.48        66
     Delayed       0.00      0.00      0.00        39
 Not Started       0.00      0.00      0.00        20
     Ongoing       0.42      0.56      0.48        61

    accuracy                           0.40       186
   macro avg       0.20      0.29      0.24       186
weighted avg       0.28      0.40      0.33       186



In [10]:

results = pd.DataFrame({
    "Original text": X_test.index.map(
        df[text_column].to_dict()
    ),
    "Clean text": X_test,
    "Actual category": y_test,
    "Predicted category": y_pred
}, index=X_test.index)

errors = results[
    results["Actual category"] != results["Predicted category"]
]

print("Всего ошибок:", len(errors))
display(errors.head(5))


Всего ошибок: 111


,Original text,Clean text,Actual category,Predicted category
611,Beneficiary list under revision,beneficiary list revision,Ongoing,Completed
236,Awaiting third-party audit clearance,awaiting third party audit clearance,Completed,Ongoing
131,Awaiting third-party audit clearance,awaiting third party audit clearance,Completed,Ongoing
488,Delay due to - contractor mobilisation issue,delay due contractor mobilisation issue,Ongoing,Completed
252,Delay due to - heavy monsoon rainfall,delay due heavy monsoon rainfall,Completed,Ongoing


### Conclusion

In Practical Work 6, I created a text preprocessing and classification pipeline using Python. I converted text to lowercase, removed punctuation, digits, and stop words, and performed tokenization.

I used TF-IDF vectorization with `min_df=2` and `max_df=0.95` to convert text into numerical features. Then, I trained a Logistic Regression classifier to predict the status of government fund utilization records.

Finally, I analyzed five classification errors and identified possible reasons, including similar text descriptions, ambiguous context, and differences between the remarks and the actual status.

This practical work helped me understand text preprocessing, TF-IDF, text classification, and error analysis.

### Analysis of Five Classification Errors

1. **Beneficiary list under revision:** The model predicted Completed instead of Ongoing. The text may describe work that is still in progress.

2. **Awaiting third-party audit clearance:** The model predicted Ongoing instead of Completed. The word “awaiting” may have influenced the prediction.

3. **Awaiting third-party audit clearance:** The model made the same incorrect prediction again. Identical text with different labels may confuse the classifier.

4. **Delay due to contractor mobilisation issue:** The model predicted Completed instead of Ongoing. The model may not have understood the meaning of the delay.

5. **Delay due to heavy monsoon rainfall:** The model predicted Ongoing instead of Completed. The model may have associated the word “delay” with an ongoing status.

**Conclusion:** The errors may be caused by ambiguous descriptions, similar words across categories, and inconsistent labels.

In [11]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

improved_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=1,
        max_df=1.0,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

improved_model.fit(X_train, y_train)
improved_pred = improved_model.predict(X_test)

print("New Accuracy:", round(
    accuracy_score(y_test, improved_pred), 4
))
print("\nNew classification report:")
print(classification_report(
    y_test, improved_pred, zero_division=0
))


New Accuracy: 0.2903

New classification report:
              precision    recall  f1-score   support

   Completed       0.41      0.42      0.41        66
     Delayed       0.24      0.23      0.24        39
 Not Started       0.05      0.10      0.07        20
     Ongoing       0.38      0.25      0.30        61

    accuracy                           0.29       186
   macro avg       0.27      0.25      0.25       186
weighted avg       0.32      0.29      0.30       186

